# Helpdesk loader — improved (henryk / U-ED-LSTM)

Sibling of `src/notebooks/loader_notebooks/normal/Helpdesk_full_loader.ipynb` that implements improvements from `improvements.md`:

- **§1 Data augmentation** — for every case whose first event is `Assign seriousness`, with probability `p = 0.5`, prepend a synthetic `Insert ticket` event. The resource is copied from the original first event; the timestamp is set equal to the original first event's timestamp (so `event_elapsed_time = 0` after re-encoding). This teaches the model the *insertion* semantics rather than the *replacement* shortcut.
- **§3 Drop the Variant index feature** — *currently disabled* (`APPLY_S3_REMOVE_VARIANT = False` in the config cell). When enabled, removes `Variant index` from `categorical_columns` to eliminate post-hoc label leakage. Flip the flag and re-run to apply.

Output pickles land in `encoded_data/improved/` so the existing `encoded_data/test_philipp/` pickles are untouched.

Note: the augmentation is applied to the *raw CSV*, so it propagates into the train/val/test splits produced by `EventLogLoader`. For the §1 success metric, evaluate on the **original** test pickle (`encoded_data/test_philipp/helpdesk_all_5_test.pkl`) so the metric remains comparable to the published numbers.

In [ ]:
import importlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

sys.path.insert(0, '../../../..')
sys.path.insert(0, '../../../../..')

import event_log_loader.new_event_log_loader
importlib.reload(event_log_loader.new_event_log_loader)
from event_log_loader.new_event_log_loader import EventLogLoader, EventLogDataset

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

SEED = 17
np.random.seed(SEED)

In [ ]:
PROJECT_ROOT = Path('../../../../..').resolve()
SRC_CSV = PROJECT_ROOT / 'data' / 'helpdesk.csv'
AUG_CSV = PROJECT_ROOT / 'encoded_data' / 'improved' / 'helpdesk_augmented.csv'
OUT_DIR = PROJECT_ROOT / 'encoded_data' / 'improved'
OUT_DIR.mkdir(parents=True, exist_ok=True)

RESULT_NAME = 'helpdesk_all'

CASE_COL = 'Case ID'
ACTIVITY_COL = 'Activity'
RESOURCE_COL = 'Resource'
TIMESTAMP_COL = 'Complete Timestamp'

SOURCE_ACTIVITY = 'Assign seriousness'
PREPEND_ACTIVITY = 'Insert ticket'
AUGMENT_PROBABILITY = 0.5

APPLY_S3_REMOVE_VARIANT = False

## 1. Read raw CSV and prepend `Insert ticket` to half of the `Assign seriousness`-start cases (§1)

In [ ]:
raw = pd.read_csv(SRC_CSV)
n_cases_total = raw[CASE_COL].nunique()
print(f'Raw event log: {len(raw)} rows | {n_cases_total} cases')

first_per_case = raw.groupby(CASE_COL, sort=False)[ACTIVITY_COL].first()
candidate_cases = first_per_case[first_per_case == SOURCE_ACTIVITY].index.tolist()
print(f'Cases starting with {SOURCE_ACTIVITY!r}: {len(candidate_cases)} '
      f'({len(candidate_cases) / n_cases_total * 100:.1f}% of total)')

rng = np.random.RandomState(SEED)
n_to_augment = int(round(len(candidate_cases) * AUGMENT_PROBABILITY))
augmented_set = set(rng.choice(candidate_cases, size=n_to_augment, replace=False))
print(f'Augmenting {n_to_augment} cases with prepended {PREPEND_ACTIVITY!r}')

In [ ]:
rows = []
n_synthetic = 0
for case_id, group in raw.groupby(CASE_COL, sort=False):
    if case_id in augmented_set:
        first_row = group.iloc[0].copy()
        first_row[ACTIVITY_COL] = PREPEND_ACTIVITY
        rows.append(first_row)
        n_synthetic += 1
    rows.extend(row for _, row in group.iterrows())

augmented_df = pd.DataFrame(rows)
assert len(augmented_df) == len(raw) + n_synthetic, 'row count off'
augmented_df.to_csv(AUG_CSV, index=False)
print(f'Wrote {AUG_CSV} | {len(augmented_df)} rows '
      f'(+{n_synthetic} synthetic {PREPEND_ACTIVITY!r})')

## 2. Encode through `EventLogLoader` (§3 gated by `APPLY_S3_REMOVE_VARIANT`)

In [ ]:
categorical_columns = [
    'Activity', 'Resource', 'Variant index',
    'seriousness', 'customer', 'product', 'responsible_section',
    'seriousness_2', 'service_level', 'service_type',
    'support_section', 'workgroup',
]

if APPLY_S3_REMOVE_VARIANT:
    categorical_columns = [c for c in categorical_columns if c != 'Variant index']
    print("§3 active: 'Variant index' removed from categorical_columns")
else:
    print("§3 disabled: 'Variant index' kept in categorical_columns (set APPLY_S3_REMOVE_VARIANT = True to drop it)")

event_log_properties = {
    'case_name': CASE_COL,
    'concept_name': ACTIVITY_COL,
    'timestamp_name': TIMESTAMP_COL,
    'date_format': '%Y/%m/%d %H:%M:%S.%f',
    'time_since_case_start_column': 'case_elapsed_time',
    'time_since_last_event_column': 'event_elapsed_time',
    'day_in_week_column': 'day_in_week',
    'seconds_in_day_column': 'seconds_in_day',
    'min_suffix_size': 5,
    'train_validation_size': 0.15,
    'test_validation_size': 0.2,
    'window_size': 'auto',
    'categorical_columns': categorical_columns,
    'continuous_columns': ['case_elapsed_time', 'event_elapsed_time', 'day_in_week', 'seconds_in_day'],
    'continuous_positive_columns': [],
}

event_log_loader = EventLogLoader(str(AUG_CSV), event_log_properties)
print('window_size:', event_log_loader.encoder_decoder.window_size)

In [ ]:
train_dataset = event_log_loader.get_dataset('train')
out = OUT_DIR / f'{RESULT_NAME}_{event_log_loader.encoder_decoder.min_suffix_size}_train.pkl'
torch.save(train_dataset, out)
print(f'Saved {out}')
print('Categorical features:', [(name, size) for name, size, _ in train_dataset.all_categories[0]])
print('Numerical features:  ', [(name, size) for name, size, _ in train_dataset.all_categories[1]])

In [ ]:
val_dataset = event_log_loader.get_dataset('val')
out = OUT_DIR / f'{RESULT_NAME}_{event_log_loader.encoder_decoder.min_suffix_size}_val.pkl'
torch.save(val_dataset, out)
print(f'Saved {out}')

In [ ]:
test_dataset = event_log_loader.get_dataset('test')
out = OUT_DIR / f'{RESULT_NAME}_{event_log_loader.encoder_decoder.min_suffix_size}_test.pkl'
torch.save(test_dataset, out)
print(f'Saved {out}')

## 3. Sanity check — confirm the activity vocabulary matches the original (so test-time evaluation can use the original pickle)

In [ ]:
ORIG_TEST = PROJECT_ROOT / 'encoded_data' / 'test_philipp' / 'helpdesk_all_5_test.pkl'
if ORIG_TEST.exists():
    orig = torch.load(ORIG_TEST, weights_only=False)
    new_act = train_dataset.all_categories[0][0]
    orig_act = orig.all_categories[0][0]
    print('Activity name->idx (improved):', new_act[2])
    print('Activity name->idx (original):', orig_act[2])
    if new_act[2] == orig_act[2]:
        print('Activity vocabularies are identical — original test pickle can be used for evaluation.')
    else:
        print('WARNING: activity index assignment differs. For honest §1 evaluation, retrain comparison must use the new test pickle, OR re-index the original.')
else:
    print(f'Original test pickle not found at {ORIG_TEST}; skipping vocabulary check.')

### Diff summary vs. `src/notebooks/loader_notebooks/normal/Helpdesk_full_loader.ipynb`

1. New augmentation cell prepends `Insert ticket` to half of the `Assign seriousness`-start cases (§1, always on).
2. `Variant index` removal (§3) is **gated by `APPLY_S3_REMOVE_VARIANT`** in the config cell; default is `False` (kept). Flip to `True` to drop it.
3. Output directory changed to `encoded_data/improved/` so the existing pickles are untouched.
4. `sys.path` adjusted for the deeper notebook location.